# Gift-Eval Ablation Analysis

This notebook analyzes the effects of a TSFM's performance across different datasets.

**Ablation Levels:**
- Level 0: Original model (no ablation)
- Level 1: Ablate layers 10, heads-all
- Level 2: Ablate layers 10-11, heads-all
- Level 3: Ablate layers 10-11-12, heads-all
- Level 4: Ablate layers 10-11-12-13, heads-all
- Level 5: Ablate layers 10-11-12-13-14, heads-all
- Level 6: Ablate layers 10-11-12-13-14-15, heads-all


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tsfm_lens.utils import apply_custom_style, normalize_by_seasonal_naive
from scipy.stats import gmean


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
HOME_DIR = os.getenv("HOME")

In [ ]:
root_dir = os.path.join(HOME_DIR, "tsfm-lens")

In [ ]:
model_name = "Chronos Bolt"

model_to_dir_mapping = {
    "TimesFM 2.5": "google-timesfm-2.5-200m-pytorch",
    "Chronos Bolt": "amazon-chronos-bolt-base",
    "Chronos": "amazon-chronos-t5-base",
    "Toto": "Datadog-Toto-Open-Base-1.0_samples-20",
}

In [ ]:
# Select the metric to analyze
SELECTED_METRIC = "eval_metrics/MASE[0.5]"
# SELECTED_METRIC = "eval_metrics/sMAPE[0.5]"

metric_pretty_name = "".join(c for c in SELECTED_METRIC.split("/")[-1] if c.isalpha())

print(f"Analyzing metric: {SELECTED_METRIC}")
print(f"Metric name: {metric_pretty_name}")


In [ ]:
save_figs = True
figs_save_dir = os.path.join(root_dir, "figs", f"gift-eval_{metric_pretty_name}", model_to_dir_mapping[model_name])
if not os.path.exists(figs_save_dir):
    os.makedirs(figs_save_dir)
if save_figs:
    print(f"Saving figures to: {figs_save_dir}")

## Load Data


In [ ]:
extra_description_str_by_key = {}
layers_srank = []
layers_srank_reverse = []

if model_name == "TimesFM 2.5":
    selected_layers_lst = [0, 5, 10, 19]
    levels_num_heads_ablated = [1, 2, 4, 6, 8, 10, 12]
    n_divisor = 16  # number of heads in a layer
    extra_description_str_by_key = {
        "Layer 0": "srank",
        "Layer 5": "reverse srank",
        "Layer 10": "reverse srank",
        "Layer 19": "best random",
    }
    layers_srank = [0]
    layers_srank_reverse = [5, 10]

elif model_name == "Chronos Bolt":
    selected_layers_lst = [0, 2, 5, 11]
    levels_num_heads_ablated = [1, 2, 4, 6, 8, 10]
    n_divisor = 12  # number of heads in a layer
    extra_description_str_by_key = {
        "Layer 0": "best random",
        "Layer 2": "reverse srank",
        "Layer 5": "best random",
        "Layer 11": "best random",
    }
    layers_srank = [0]
    layers_srank_reverse = [2]

elif model_name == "Toto":
    selected_layers_lst = [0, 2, 9, 11]
    levels_num_heads_ablated = [1, 2, 4, 6, 8, 10]
    n_divisor = 12  # number of heads in a layer
    extra_description_str_by_key = {
        "Layer 0": "srank",
        "Layer 2": "best random",
        "Layer 9": "reverse srank",
        "Layer 11": "srank",
    }
    layers_srank = [0, 11]
    layers_srank_reverse = [9]
else:
    raise ValueError(f"Model {model_name} not supported yet")

# Build ablation files dictionary for all selected layers
ablated_files_dict = {}
ablation_meaning_str_dict = {}
ablation_level_meaning_str_dict = {}
ablation_level_meaning_alternative_str_dict = {}

ablation_meaning_str = "Ablation of Heads Across Layers"
ablation_level_meaning_str = "Pct of Heads Ablated per Layer"
ablation_level_meaning_alternative_str = "Number of Heads Ablated per Layer"

for selected_layers in selected_layers_lst:
    selected_layers_str = str(selected_layers)
    key = f"Layer {selected_layers_str}"

    ablation_meaning_str_dict[key] = f"Ablation of Heads in Layer {selected_layers_str}"
    ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated"
    ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated"

    ablation_files = {
        0: "original_all_results.csv",
        **{n: f"head_layers_{selected_layers_str}_heads-{n}_all_results.csv" for n in levels_num_heads_ablated},
        n_divisor: f"head_layers_{selected_layers_str}_heads-all_all_results.csv",
    }
    ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in ablation_files.items()}
    ablated_files_dict[key] = ablation_files

In [ ]:
ablation_files

In [ ]:
ablated_files_dict.keys()

In [ ]:
random_rseeds_lst = [
    123,
    99,
    42,
    688,
    22,
    33,
    56,
    78,
    357,
    234,
    567,
    890,
    1356,
    2233,
    3566,
    4891,
    222,
    333,
    444,
    555,
    777,
]
random_metrics_dir_dict = {
    rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"rseed-{rseed}")
    for rseed in random_rseeds_lst
}

srank_rseeds_lst = [99, 42]
srank_metrics_dir_dict = {
    rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"srank_rseed-{rseed}")
    for rseed in srank_rseeds_lst
}

srank_reverse_rseeds_lst = [99, 42]
srank_reverse_metrics_dir_dict = {
    rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"srank_reverse_rseed-{rseed}")
    for rseed in srank_reverse_rseeds_lst
}

print(f"Random head selection metrics directory: {random_metrics_dir_dict}")
print(f"Stable rank (low to high) head selection metrics directory: {srank_metrics_dir_dict}")
print(f"Stable rank (high to low) head selection metrics directory: {srank_reverse_metrics_dir_dict}")


In [ ]:
layers_srank_str = [f"Layer {layer}" for layer in layers_srank]
layers_srank_reverse_str = [f"Layer {layer}" for layer in layers_srank_reverse]
print(f"Layers srank: {layers_srank_str}")
print(f"Layers srank reverse: {layers_srank_reverse_str}")

In [ ]:
# Load and combine CSV files across ablation levels and rseeds
# For each level, compute geometric mean across datasets for each rseed,
# then select the rseed with the lowest geometric mean
best_filepaths_by_key = {}
data_by_key = {}

for key, ablation_files in ablated_files_dict.items():
    data_by_key[key] = {}
    best_filepaths_by_key[key] = {}
    curr_data_frame = {}

    if key in layers_srank_str:
        metrics_dir_dict = srank_metrics_dir_dict
    elif key in layers_srank_reverse_str:
        metrics_dir_dict = srank_reverse_metrics_dir_dict
    else:
        metrics_dir_dict = random_metrics_dir_dict

    for level, filepath in ablation_files.items():
        rseed_data = {}
        available_rseeds = []

        # Load data for each rseed
        for rseed, metrics_dir in metrics_dir_dict.items():
            df_filepath = os.path.join(metrics_dir, filepath)
            if not os.path.exists(df_filepath):
                print(f"Level {level}, rseed {rseed}: Skipping (not found)")
                continue

            df = pd.read_csv(df_filepath)
            if len(df) != 97:
                print(f"Level {level}, rseed {rseed}: Skipping (incomplete)")
                continue

            print(f"Level {level}, rseed {rseed}: Loading")
            df["ablation_level"] = level
            rseed_data[rseed] = (df, df_filepath)
            available_rseeds.append(rseed)

        if not rseed_data:
            print(f"Level {level}: No data found, skipping")
            continue

        # Compute geometric mean for each rseed and select best
        rseed_gmeans = {}
        for rseed, (df, df_filepath) in rseed_data.items():
            selected_metric_gmean = gmean(df[SELECTED_METRIC])
            rseed_gmeans[rseed] = {
                f"gmean_{SELECTED_METRIC}": selected_metric_gmean,
                "filepath": df_filepath,
                "df": df,
            }

        # Select rseed with lowest geometric mean
        best_rseed = min(rseed_gmeans.keys(), key=lambda r: rseed_gmeans[r][f"gmean_{SELECTED_METRIC}"])
        best_df = rseed_gmeans[best_rseed]["df"]
        best_filepath = rseed_gmeans[best_rseed]["filepath"]

        best_filepaths_by_key[key][level] = {
            "best_rseed": best_rseed,
            "best_filepath": best_filepath,
            "available_rseeds": available_rseeds,
            f"gmean_{SELECTED_METRIC}": rseed_gmeans[best_rseed][f"gmean_{SELECTED_METRIC}"],
            "num_rseeds_evaluated": len(rseed_data),
        }

        curr_data_frame[level] = best_df
        print(
            f"Level {level}: Best rseed={best_rseed} (gmean={rseed_gmeans[best_rseed][f'gmean_{SELECTED_METRIC}']:.6f}) from {len(rseed_data)} rseeds"
        )

    all_data_for_key = pd.concat(curr_data_frame.values(), ignore_index=True)
    data_by_key[key] = all_data_for_key
    print(f"\nTotal: {len(all_data_for_key)} rows, {all_data_for_key['dataset'].nunique()} unique datasets")

In [ ]:
# Structure: best_filepaths_by_key[key][level] = {"rseed": ..., "filepath": ..., "overall_gmean": ..., "metric_gmeans": {...}}
print("Example of stored filepath information:")
print("=" * 80)
example_key = list(best_filepaths_by_key.keys())[1]
for key in [example_key]:  # Just show first key as example
    print(f"\nKey: {key}")
    for level in list(best_filepaths_by_key[key].keys()):
        info = best_filepaths_by_key[key][level]
        print(f"  Level {level}:")
        print(f"    Best rseed: {info['best_rseed']}, {info['best_filepath']}")
        print(f"    {SELECTED_METRIC} gmean: {info[f'gmean_{SELECTED_METRIC}']:.6f}")
        print(f"    {info['num_rseeds_evaluated']} rseeds evaluated: {info['available_rseeds']}")


In [ ]:
rng = np.random.default_rng(333)
layers_to_ablate = [10]
ablate_n_heads_per_layer = 10
heads_to_ablate = []
for layer in layers_to_ablate:
    heads_to_ablate_for_layer = rng.choice(list(range(12)), size=ablate_n_heads_per_layer, replace=False)
    heads_to_ablate.extend([(layer, head) for head in heads_to_ablate_for_layer])  # type: ignore
print(heads_to_ablate)

## Optional: Normalize by Seasonal Naive Baseline


In [ ]:
# Configuration: Set to True to normalize metrics by seasonal naive baseline
NORMALIZE_BY_SEASONAL_NAIVE = True

print(f"Normalization by seasonal naive baseline: {NORMALIZE_BY_SEASONAL_NAIVE}")


In [ ]:
if NORMALIZE_BY_SEASONAL_NAIVE:
    for key in data_by_key.keys():
        all_data = data_by_key[key]
        print("\nLoading seasonal naive baseline...")
        seasonal_naive_path = os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv")
        seasonal_naive_df = pd.read_csv(seasonal_naive_path)
        print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets")

        print("\nNormalizing data by seasonal naive baseline...")
        all_data = normalize_by_seasonal_naive(all_data, seasonal_naive_df)
        data_by_key[key] = all_data
else:
    print("\nSkipping normalization.")

## Display Available Metrics


In [ ]:
# Display available metrics
metric_columns = [col for col in all_data.columns if "eval_metrics" in col]
print("Available metrics:")
for i, metric in enumerate(metric_columns, 1):
    print(f"{i}. {metric}")


## 1. Box Plot: Overall Performance Across Ablation Levels


In [ ]:
# Prepare data for box plot
box_data_by_key = {}
for key in data_by_key.keys():
    box_data = data_by_key[key][["dataset", "ablation_level", SELECTED_METRIC]].copy()
    # Remove any infinite or NaN values
    box_data = box_data.replace([np.inf, -np.inf], np.nan).dropna()
    box_data_by_key[key] = box_data


## 2. Line Plot: Median Performance Trend


In [ ]:
# Calculate median and percentiles for each ablation level and key
stats_by_level_by_key = {}
for key in data_by_key.keys():
    box_data = box_data_by_key[key]
    stats_by_level = (
        box_data.groupby("ablation_level")[SELECTED_METRIC]
        .agg(
            [
                "median",
                "mean",
                lambda x: gmean(x),
                lambda x: np.exp(np.std(np.log(x)) / np.sqrt(len(x))),  # geometric standard error (scale factor)
                lambda x: x.quantile(0.25),
                lambda x: x.quantile(0.75),
            ]
        )
        .rename(columns={"<lambda_0>": "geom_mean", "<lambda_1>": "geom_ste", "<lambda_2>": "q25", "<lambda_3>": "q75"})
        .reset_index()
    )
    stats_by_level_by_key[key] = stats_by_level

In [ ]:
single_layer_colors = plt.cm.get_cmap("cividis")(np.linspace(0.1, 0.95, len(data_by_key.keys())))
colors = list(single_layer_colors)

In [ ]:
data_by_key.keys()

In [ ]:
stats_by_level

In [ ]:
stats_by_level

In [ ]:
# Create line plot with error bands
fig, ax = plt.subplots(figsize=(5, 4))

# Plot lines for each key
for i, key in enumerate(data_by_key.keys()):
    extra_description_str = extra_description_str_by_key[key]
    stats_by_level = stats_by_level_by_key[key]

    # Get baseline values

    if i == 0:
        baseline_geom_mean = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "geom_mean"].values[0]
        # Plot baseline reference lines
        ax.axhline(
            y=baseline_geom_mean,
            color="red",
            linestyle="--",
            linewidth=2,
            label=f"Baseline ({baseline_geom_mean:.3f})",
            alpha=0.7,
            zorder=4,
        )
        ax.text(
            0,
            baseline_geom_mean * 0.994,
            f"Baseline ({baseline_geom_mean:.3f})",
            color="red",
            fontsize=8,
            fontweight="bold",
            va="top",
            ha="left",
            zorder=100,
        )

        ax.axhline(y=1.01 * baseline_geom_mean, color="orangered", linestyle="--", linewidth=2, alpha=0.7)
        ax.text(
            0,
            (1.01 * baseline_geom_mean) * 1.004,
            "Baseline + 1%",
            color="orangered",
            fontsize=8,
            fontweight="bold",
            va="bottom",
            ha="left",
            zorder=100,
        )

    color = colors[i]
    # Plot geometric mean line with error envelope
    ax.plot(
        stats_by_level["ablation_level"] / n_divisor * 100,
        stats_by_level["geom_mean"],
        marker="^",
        markerfacecolor="None",
        markeredgecolor=color,
        markeredgewidth=2,
        linewidth=2.5,
        markersize=7,
        label=f"{key} ({extra_description_str})",
        color=color,
        linestyle="-",
        alpha=0.8,
    )

    ax.fill_between(
        stats_by_level["ablation_level"] / n_divisor * 100,
        stats_by_level["geom_mean"] * (stats_by_level["geom_ste"] ** 0.33),
        stats_by_level["geom_mean"] / (stats_by_level["geom_ste"] ** 0.33),
        alpha=0.05,
        color=color,
        zorder=-1,
    )

# Formatting
ax.set_xlabel(ablation_level_meaning_str + " (%)", fontweight="bold")
ax.set_ylabel(f"{metric_pretty_name} (Geom Mean)", fontweight="bold")
# ax.set_title(f"{ablation_meaning_str} ({model_name})", fontweight="bold", pad=20)
ax.set_title(model_name, fontweight="bold", pad=10)
ax.grid(True, alpha=0.2)
ax.legend(loc="upper left", frameon=True)

# Set x-axis ticks
all_levels = set()
for stats_by_level in stats_by_level_by_key.values():
    all_levels.update(stats_by_level["ablation_level"].unique())
xticks = sorted([x / n_divisor * 100 for x in all_levels])
ax.set_xticks(xticks)
ax.set_xticklabels([f"{int(x)}" if int(x) == x else f"{x:.1f}" for x in xticks], rotation=0)

# Set y-axis limits
ax.set_ylim(0.98 * baseline_geom_mean, 1.06 * baseline_geom_mean)

plt.tight_layout()
save_path = os.path.join(figs_save_dir, f"{ablation_meaning_str}_line_plot_{model_name}.pdf")
if save_figs:
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved line plot to: {save_path}")
plt.show()

In [ ]:
# Calculate and display percentage change from baseline for each key
baseline_median = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "median"].values[0]
baseline_geom_mean = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "geom_mean"].values[0]

for key in data_by_key.keys():
    stats_by_level = stats_by_level_by_key[key]

    print(f"\nPercentage Change from Baseline (Level 0) for {key}:")
    print("=" * 70)
    for _, row in stats_by_level.iterrows():
        num_heads_ablated = int(row["ablation_level"])
        pct_heads_ablated = num_heads_ablated / n_divisor * 100

        median_change = ((row["median"] - baseline_median) / baseline_median) * 100
        curr_geom_mean = row["geom_mean"]
        geom_mean_change = ((curr_geom_mean - baseline_geom_mean) / baseline_geom_mean) * 100
        print(
            f"{num_heads_ablated} heads ({pct_heads_ablated:.1f}%) ablated: Median {median_change:+.2f}%, Geom Mean {geom_mean_change:+.2f}%"
        )
        print(f"==== Current Geom Mean: {curr_geom_mean:.3f}")
